# 01 · Inference — Tokens, Attention & Generation

**Runs on: Local CPU** (no GPU required for this 135M model)

By the end of this notebook you will understand:
- How text is split into tokens (subwords)
- What `input_ids` and `attention_mask` are
- How `generate()` works and what each parameter does
- Why the base model *cannot* follow instructions (and why we need fine-tuning)

## Setup

In [ ]:
# If running for the first time, uncomment:
# %pip install transformers accelerate torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Tokenizer & Model

A **tokenizer** converts text ↔ token ids.  
A **model** takes token ids and predicts the *next* token (causal LM).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval()

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded: {num_params:.0f}M parameters")
print(f"Architecture: {type(model).__name__}")

## 2. What Is a Token?

LLMs do not process characters or words — they process **subword tokens**.  
The vocabulary here is ~49k tokens. Each word maps to 1–3 tokens typically.

In [ ]:
text = "Hello, I am learning about large language models!"
token_ids = tokenizer.encode(text)

print(f"Input text:  '{text}'")
print(f"Num tokens:  {len(token_ids)}")
print(f"Token ids:   {token_ids}")
print()

# Decode each token individually to see the subwords
print("id  → token")
print("-" * 30)
for tid in token_ids:
    token_str = tokenizer.decode([tid])
    print(f"{tid:6d}  → '{token_str}'")

In [ ]:
# Some tokens are interesting — notice how uncommon words split into multiple tokens
words = ["cat", "cats", "Transformer", "tokenization", "supercalifragilistic"]
for w in words:
    toks = tokenizer.encode(w, add_special_tokens=False)
    decoded = [tokenizer.decode([t]) for t in toks]
    print(f"'{w}' → {decoded} ({len(toks)} token{'s' if len(toks)>1 else ''})")

## 3. The Tensors the Model Sees

`tokenizer(text, return_tensors='pt')` returns a dict with two tensors:
- **`input_ids`**: the token id sequence, shape `[batch_size, seq_len]`
- **`attention_mask`**: 1 where a token exists, 0 for padding, same shape

In [ ]:
inputs = tokenizer(text, return_tensors="pt")
print("Keys in tokenizer output:", list(inputs.keys()))
print(f"input_ids shape:       {inputs['input_ids'].shape}  (batch=1, seq_len={inputs['input_ids'].shape[1]})")
print(f"attention_mask shape:  {inputs['attention_mask'].shape}")
print(f"input_ids values:      {inputs['input_ids'][0].tolist()}")

In [ ]:
# Run a forward pass — the model outputs logits (raw scores for each possible next token)
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(f"logits shape: {logits.shape}  (batch=1, seq_len={logits.shape[1]}, vocab_size={logits.shape[2]})")

# The last position's logits predict the NEXT token
next_token_logits = logits[0, -1, :]
next_token_id = next_token_logits.argmax().item()
print(f"Most likely next token: id={next_token_id}, text='{tokenizer.decode([next_token_id])}'")

## 4. Text Generation

`.generate()` automates the loop: predict next token → append → predict next token → repeat.  
Default is **greedy decoding**: always pick the highest-probability token.

In [ ]:
def generate(prompt, max_new_tokens=60, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

prompt = "The history of artificial intelligence began"
print(f"Prompt: '{prompt}'")
print(f"Greedy output: {generate(prompt)}")

## 5. Generation Parameters — Explore the Tradeoffs

| Parameter | What it does |
|---|---|
| `do_sample=False` | Greedy — always pick top-1 (deterministic, repetitive) |
| `do_sample=True` | Sample from the distribution (add randomness) |
| `temperature` | Scales logits before softmax. < 1 = sharper (more confident), > 1 = flatter (more random) |
| `top_p` | Nucleus sampling: only sample from tokens covering cumulative prob `p` |
| `top_k` | Only sample from the top-k highest probability tokens |

In [ ]:
prompt = "Once upon a time in a faraway land"

configs = [
    {"label": "Greedy (deterministic)",         "do_sample": False},
    {"label": "Sample, temp=1.0 (default)",      "do_sample": True, "temperature": 1.0},
    {"label": "Sample, temp=0.3 (conservative)", "do_sample": True, "temperature": 0.3},
    {"label": "Sample, temp=1.5 (creative)",     "do_sample": True, "temperature": 1.5},
    {"label": "top_p=0.9 (nucleus)",             "do_sample": True, "temperature": 1.0, "top_p": 0.9},
]

torch.manual_seed(42)
for cfg in configs:
    label = cfg.pop("label")
    out = generate(prompt, max_new_tokens=40, **cfg)
    print(f"[{label}]")
    print(f"  {out}")
    print()

## 6. Chat Templates — And Why the Base Model Can't Follow Instructions

Chat models are trained with a **structured conversation format** using special tokens to delimit turns.  
The base model has no concept of this format — it just predicts the next token as if continuing plain text.

`tokenizer.apply_chat_template()` formats a list of messages into this structured string.

In [ ]:
messages = [
    {"role": "user", "content": "What is the capital of France? Answer in one word."}
]

# The BASE tokenizer has no chat_template — borrow it from the Instruct variant
# (they share the same vocabulary; the template is just a formatting string)
if tokenizer.chat_template is None:
    from transformers import AutoTokenizer as _T
    tokenizer.chat_template = _T.from_pretrained(
        "HuggingFaceTB/SmolLM2-135M-Instruct"
    ).chat_template
    print("Chat template loaded from Instruct tokenizer.")

# Format using the chat template
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True  # adds the assistant turn starter
)
print("Formatted prompt (what the model actually sees):")
print(repr(formatted))
print()
print("Rendered:")
print(formatted)

In [ ]:
# Now ask the BASE model to follow this instruction
# Expect: it will NOT follow the instruction — it continues with plausible next tokens
print("Base model response (expect non-instruction-following):")
print(generate(formatted, max_new_tokens=50, do_sample=True, temperature=0.7))

In [ ]:
# For comparison, load the Instruct model (fine-tuned version)
# This downloads ~270MB — skip if bandwidth is tight
from transformers import AutoTokenizer, AutoModelForCausalLM

INSTRUCT_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
instruct_tok = AutoTokenizer.from_pretrained(INSTRUCT_ID)
instruct_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_ID, torch_dtype=torch.float32)
instruct_model.eval()

formatted_instruct = instruct_tok.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inp = instruct_tok(formatted_instruct, return_tensors="pt")
with torch.no_grad():
    out = instruct_model.generate(
        **inp, max_new_tokens=50, do_sample=True, temperature=0.3,
        pad_token_id=instruct_tok.eos_token_id
    )
new_toks = out[0][inp["input_ids"].shape[1]:]
print("Instruct model response (fine-tuned — should follow the instruction):")
print(instruct_tok.decode(new_toks, skip_special_tokens=True))

## Summary

What you just learned:

1. **Tokenization** splits text into subword units (vocabulary of ~49k tokens)
2. The model input is `[batch, seq_len]` integer tensors; output is `[batch, seq_len, vocab_size]` logits
3. **Generation** = repeatedly argmax/sample the next token logits
4. **Temperature / top_p / top_k** trade off diversity vs coherence
5. The **base model** cannot follow instructions — it just predicts plausible continuations
6. The **Instruct model** was fine-tuned on (instruction, response) pairs — that's what we'll do in notebook 03!

**Next**: [`02_explore_dataset.ipynb`](02_explore_dataset.ipynb) — look at the training data format.